# 02 - Chunking comparison: structure-aware vs naive

**Question of this notebook:** does it matter *how* we cut the law into chunks?

- **Structure-aware chunking** (what we use, built in notebook 01): one chunk per article paragraph, recital or annex point. Every chunk is a complete legal statement, starts with its heading (`Article 7 – Conditions for consent`) and knows exactly which provision it is.
- **Naive chunking** (the baseline): glue the whole regulation together into one long text and cut it every 850 characters into pieces of 1000 characters (150 characters overlap), no matter where sentences or articles end. Each piece is labelled with the provision in which it *starts*, because that is the only label we can give it.

We build both, index both with the same embedding model and the same hybrid retrieval, and measure how often the right provision comes back (hit@k, explained in notebook 06).

In [1]:
import os
import warnings

# Silence noise before anything is imported (deprecation notices, HF logs).
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_VERBOSITY"] = "error"

import json  # noqa: E402
import sys  # noqa: E402
import textwrap  # noqa: E402
from pathlib import Path  # noqa: E402

import pandas as pd  # noqa: E402

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # run from the project root so data/ and eval/ paths work
sys.path.insert(0, str(Path.cwd()))  # so that `import eval...` works

from assistant.chunking import naive_pieces  # noqa: E402
from assistant.parse import read_chunks  # noqa: E402

chunks = read_chunks(Path("data/chunks.jsonl"))
pieces = naive_pieces(chunks, size=1000, overlap=150)
naive = [p for p, _ in pieces]
print(f"structured: {len(chunks)} chunks, naive: {len(naive)} pieces")
print("median length structured:", int(pd.Series([len(c.text) for c in chunks]).median()), "chars")
print("median length naive:     ", int(pd.Series([len(p.text) for p in naive]).median()), "chars")

structured: 3044 chunks, naive: 2406 pieces
median length structured: 474 chars
median length naive:      1000 chars


## A structured chunk

One paragraph, complete, with its heading. The label (regulation, article, paragraph) is exact, so a citation built from it is exact.

In [2]:
by_id = {c.id: c for c in chunks}
print(by_id["GDPR-en-article-7-2"].text)

Article 7 – Conditions for consent
(2) If the data subject's consent is given in the context of a written declaration which also concerns other matters, the request for consent shall be presented in a manner which is clearly distinguishable from the other matters, in an intelligible and easily accessible form, using clear and plain language. Any part of such a declaration which constitutes an infringement of this Regulation shall not be binding.


## A naive piece

Below is one naive piece from the English GDPR. Look at three things: where it **starts** (in the middle of a word), where it **ends** (in the middle of a sentence), and which provisions its text actually contains compared with the label it carries.

In [3]:
def show(piece, covered):
    label = f"{piece.regulation} Art. {piece.number}({piece.paragraph})"
    print("label (where it starts):", label)
    print("text touches:", ", ".join(f"Art. {k[2]}({k[3]})" for k in covered))
    print("starts with:", repr(piece.text[:60]))
    print("ends with:  ", repr(piece.text[-60:]))


example = next(
    (p, cov)
    for p, cov in pieces
    if p.regulation == "GDPR"
    and p.language == "en"
    and p.number == "6"
    and {k[2] for k in cov} == {"6", "7"}
)
show(*example)

label (where it starts): GDPR Art. 6(4)
text touches: Art. 6(4), Art. 7(1), Art. 7(2), Art. 7(3)
starts with: 'cular whether special categories of personal data are proces'
ends with:   'ringement of this Regulation shall not be binding.\n(3) The d'


The piece is labelled **GDPR Art. 6(4)** (lawfulness, further processing), but about 60 % of its text is **Article 7** (conditions for consent), and it cuts Art. 7(3) off after "The d". If retrieval picks this piece for a consent question, the citation would say "Art. 6(4)": the answer might be right, but the citation would point to the wrong article. With structured chunks that cannot happen.

## Measured on the gold set

Both runs: 40 gold questions (35 in scope, 5 out of scope), `top_k = 6`, BM25 weight 0.4 / vector weight 0.6, `multilingual-e5-small`. The numbers come from `eval/runs/*.json`, written by:

```
uv run python eval/run_eval.py --retrieval-only --chunking structured
uv run python eval/run_eval.py --retrieval-only --chunking naive
```

In [4]:
runs = {
    name: json.loads(Path(f"eval/runs/{name}.json").read_text()) for name in ("structured", "naive")
}


def pct(m):
    return "n/a" if m is None else f"{m['value']:.1%} (n={m['n']})"


pd.DataFrame(
    {
        name: {
            "hit@k article (by label)": pct(r["summary"]["hit_article"]),
            "hit@k paragraph (by label)": pct(r["summary"]["hit_paragraph"]),
            "hit@k article (by text covered)": pct(r["summary"]["hit_article_text"]),
            "indexed documents": r["meta"]["indexed_documents"],
        }
        for name, r in runs.items()
    }
)

,structured,naive
hit@k article (by label),82.9% (n=35),54.3% (n=35)
hit@k paragraph (by label),64.3% (n=28),32.1% (n=28)
hit@k article (by text covered),n/a,57.1% (n=35)
indexed documents,3638,2406


"By label" is what a citation would show. "By text covered" gives naive chunking the benefit of the doubt: it counts a hit if the expected article's text is *anywhere* inside a retrieved piece, even under a wrong label.

Which questions did the two methods disagree on?

In [5]:
s = {r["id"]: r for r in runs["structured"]["records"]}
n = {r["id"]: r for r in runs["naive"]["records"]}
gold = {g["id"]: g for g in map(json.loads, Path("eval/gold.jsonl").read_text().splitlines())}
rows = [
    {
        "id": i,
        "question": textwrap.shorten(gold[i]["question"], 60),
        "structured": s[i]["hit_article"],
        "naive (label)": n[i]["hit_article"],
        "naive (text)": n[i]["hit_article_text"],
    }
    for i in s
    if not s[i]["out_of_scope"] and s[i]["hit_article"] != n[i]["hit_article"]
]
pd.DataFrame(rows)

,id,question,structured,naive (label),naive (text)
0,q03,How do I find out whether my AI system counts ...,True,False,False
1,q04,"Gilt ein KI-System, das Bewerbungen automatisc...",True,False,False
2,q08,"Muss ich Nutzern sagen, dass sie mit einem Cha...",True,False,False
3,q11,How high can the fines be for using a prohibit...,True,False,False
4,q12,Welche Dokumentation müssen Anbieter von KI-Mo...,True,False,True
5,q13,Do companies that use AI have to make sure the...,True,False,False
6,q17,Can I make a company delete the personal data ...,True,False,False
7,q28,Dürfen personenbezogene Daten in ein Land auße...,True,False,False
8,q31,A bank's AI system rejected my loan applicatio...,True,False,False
9,q34,Muss der Betreiber eines Hochrisiko-KI-Systems...,True,False,False


## What this shows, and what it does not

- Structured chunking found the expected article for **29 of 35** questions (82.9%), naive chunking for **19 of 35** (54.3%). At paragraph level the gap is 64.3% vs 32.1% (n = 28).
- Naive never won a question that structured lost: all 10 disagreements above go the same way. With 10 of 10, a simple sign test says this is very unlikely to be chance (p ≈ 0.002) on this question set.
- Even when we count text hidden under the wrong label, naive only reaches 57.1%. So the gap is not just about labels: pieces that mix two articles and start mid-sentence are also harder to *find*.
- **Caveats.** 35 questions is a small sample: one question is ~3 percentage points. We wrote the questions ourselves while knowing the system, so they may favour clean article-level answers, which suits structured chunking. Naive chunking was also not tuned (other sizes, splitting at sentence ends), so this compares our design with the simplest baseline, not with the best possible alternative.

**Try it:** change `size` and `overlap` in the first code cell and look at how the example piece changes. (Re-running the naive evaluation with other sizes needs a code change in `run_eval.py`; it rebuilds `data/chroma_naive` automatically.)